<!-- ## Calculating Laplace Transform with Laguerre Quadrature
Our modelling technique (as well as CONTIN) performs the Laplace inversion by calculating the Laplace transform for some given parameters, and then optimizing those parameters to observation data. 

$$
g_1(\tau) = \int\limits_{0}^{\infty} G{\left(\Gamma \right)} e^{- \Gamma \tau}\, d\Gamma
$$


CONTIN is a discretized model so it's solution to calculating the integral is basically to model $G$ as a mixture of Dirac delta functions. Our solution models $G$ as a mixture of normal distributions, which has the added benefit of being a continuous distribution, and thus more intepretable and physically meaningful. The fact that the normal distribution model allows for the integral to be expressed in closed form is a lucky break, which is not something that can be expected when using arbitrary distributions including ones that are more desirable for their physical feasibility such as a mixture of log-normal distributions. Technically, it also employs a quadrature, since it uses the error function `erf`, which is calculated using a numerical method.

Instead of relying on luck for obtaining a closed-form expression, we can use numerical integration. In particular, Gauss-Laguerre quadrature allows us to do this in an accurate and efficient way. The quadrature states that

$$
\int\limits_{0}^{\infty} f(x) e^{-x} dx \approx \sum_{i=1}^{n}{w_i f(x_i)}
$$

Where $x_i$ are the roots of the n-th Laguerre polynomial and $w_i$ are the Gaussian quadrature weights which allow for the above approximation to be an equality for polynomials of degree $2n-1$. Since we are working with multi-angle data, we should also transform our integral to account for the fact that $\Gamma = D q^2$ where $q$ has an angular dependence:
$$
g_1(\tau) = \int\limits_{0}^{\infty} P{\left(D \right)} e^{- D q^2 \tau}\, dD = \int\limits_{0}^{\infty} \left[P{\left(D \right)} e^{(1 - q^2 \tau) D}\right] e^{-D}\, dD
$$



 It is apparent that this can be used- though not necessarily efficiently so- for our integral because the exponent term in the laplace transform contains the original domain variable and the $s$ variable, or in our case, $\Gamma$ and $\tau$ respectively. Since $\tau$ is simply a set of delay times that correspond to our observed values, we can naively redefine $G$ as a function of $\Gamma \tau$ for each concrete value of $\tau$ and then recalcuate it at the Laguerre root points. We could actually take this further to then make not only usable but efficient as well, such that the calculation of this transformed function relies on $G$ calculated at the same $n$ points, rather than having to do so for each separate value of $\tau$.

Firsly, since we are working with multi-angle data, we should also transform our integral to account for the fact that $\Gamma = D q^2$ where $q$ has an angular dependence:



We can thus modify the integral of interest to fit this form and moreover, allow for calculating the $G$ at only $n$ points 



 -->
redacted notes

## Approximate Laplace Transform with Laguerre Quadrature
Gaussian quadrature using the Laguerre polynomials gives us a way to evaluate an integral of a certain form:
$$
\int\limits_{0}^{\infty} f(x) e^{-x} dx \approx \sum_{i=1}^{n}{w_i f(x_i)}
$$
Where $x_i$ is the $i$-th root of the $n$-th Laguerre polynomial and $w_i$ is its corresponding quadrature weight. The approximation is an equality for polynomials of degree $2n-1$ - which is called the _degree of exactness_.

We can then adapt the integral for the DLS $g_1$ function to fit into this quadrature:

$$
g_1(q, \tau) = \int\limits_{0}^{\infty} G{\left(\Gamma \right)} e^{- \Gamma \tau}\, d\Gamma = \int\limits_{0}^{\infty} P{\left(D \right)} e^{- D q^2 \tau}\, dD = \int\limits_{0}^{\infty} \left[P{\left(D \right)} e^{(1 - q^2 \tau) D}\right] e^{-D}\, dD
$$

Thus we can approximate $g_1$ with the quadrature $g_{1_\text{GL}}$, which we can substitute into the overall model for $g_2$:
$$
g_1(q, \tau) \approx g_{1_\text{GL}}(q, \tau) = \sum_{i=1}^{n}{w_i \cdot P{\left(x_i \right)} e^{(1 - q^2 \tau) x_i}}
$$

### Modify Quadrature for Domain Scaling
One problem that remains is that if the scale of $P$ is very small (which we expect it to be), then evaluating it at the points $x_i$ may yield very small values and cause inaccuracy. I think this due to numerical precision, rather than due to some mathematical property. In that case, we can scale the domain as follows:

let $c$ be a constant and $D = c \cdot x$ (and thus also $ dD = c \cdot dx $):

$$
g_1(q, \tau) = \int\limits_{0}^{\infty} P{\left(c x \right)} e^{- c x q^2 \tau}\, c dx = c \cdot \int\limits_{0}^{\infty} \left[P{\left(c x \right)} e^{(1 - q^2 \tau) c x}\right] e^{-x}\, dx
$$

So we can define scaled root points $ D_i = c x_i$ and our quadrature becomes

$$
g_{1_\text{GL}}(q, \tau) = c \cdot \sum_{i=1}^{n}{w_i \cdot P{\left(D_i \right)} e^{(1 - q^2 \tau) D_i}}
$$

### Comparison to Discrete Models
CONTIN style models provide an interesting comparison because they use a discrete model and do not attempt to represent a continuous distribution at all.
In the set up, there is a user defined, fixed grid of $m$ values for the diffusion coefficient $D_j$ and corresponding contribution $P'(D_j)$ are fitted to the data. It can also be thought of modeling the distribution as a mixture of dirac functions.
$$
g_1'(q, t) = \sum_{j=1}^m {P'(D_j) \cdot e^{-D_j q^2 t}} = \sum_{j=1}^m {\left(P'(D_j) \cdot \int\limits_{0}^{\infty} \delta\left(D - D_j\right) e^{-D q^2 t}\, dD \right)}
$$

It actually looks very much like our quadrature, except that (1) the grid is arbitrary and (2) the sum is unweighted.

### Comparison to Our Current Model
In our current approach, we model the distribution of diffusion coefficients as a mixture of Gaussians. That allowed us to evaluate the $g_1$ integral with a function that includes an the error function $\textbf{erf}$, which is implemented as a quadrature of its own.

### Comparison to Current Model
1) the quadrature and current approach outputs differed by a small epsilon for large enough laguerre degree $n$ (empirically $n=60$ seems sufficient). Optimizations gave equivalent results for fitting parameters ($\alpha_k, \mu_k, \sigma_k$) as well. I think this confirms that the quadrature works as a drop in replacement for the GMM, and implies that the quadrature can be used for other mixture models, such as ones using log-normal distributions instead of normal ones.
2) The quadrature requires double precision. even just getting the Laguerre root points and quadrature weights for $n>25$ returned vectors containing NaN's. The trust region optimization method uses 64 bit precision by default anyway, so it hasn't been noticeable, but could be affect performance.
3) Overall performance, given both approaches us double precision, is similar - optimizations take about the same amount of (wall) time

In [1]:
import jax
jax.config.update("jax_enable_x64", True)  # Enable 64-bit precision
import orthax

ERROR:2025-09-02 00:00:03,078:jax._src.xla_bridge:444: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/gfertt/.local/lib/python3.10/site-packages/jax/_src/xla_bridge.py", line 442, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/gfertt/.local/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 324, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/gfertt/.local/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 281, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 100; cuGetErrorName failed. This probably means that JAX was unable to load the CUDA libraries.


In [5]:
# quadrature based model
import orthax
import jax
import jax.numpy as jnp
import functools
import pandas as pd

jax.config.update("jax_enable_x64", True)  # Enable 64-bit precision

SCALING_CONST = 2.45e-12
relative_scaling = 1.3e-14 / SCALING_CONST

In [6]:
def scatter_vector(theta, lambda_0=633e-9, n=1.33, radians=False):
    """
    theta - scatter angle
    lambda_0 - lsder wavelength in meters
    n - refractive index of water
    """
    if not radians:
        theta = jnp.radians(theta)
    return (4 * jnp.pi * n/lambda_0) * jnp.sin(theta / 2)

def diffusion_coef(r, k_B=1.38e-23, T=298.15, eta=0.00089):
    """
    r - particle radius
    k_b - Boltzmann constant (J/K)
    T - Temperature (K)
    eta - Viscosity of water at room temerature (Pa*s)
    """
    return k_B * T / (6 * jnp.pi * eta * r)

def normal_to_lognormal_params(mu, sigma):
    sigma_ln_sq = jnp.log(1 + (sigma / mu)**2)
    mu_ln = jnp.log(mu) - (sigma_ln_sq / 2)
    sigma_ln = jnp.sqrt(sigma_ln_sq)
    return (mu_ln, sigma_ln)

def prep_data(datafile):
    df = pd.read_csv(datafile, delimiter='\t', header=None)
    df = df.iloc[1:] # remove strange first point
    d = jnp.array(df.to_numpy())

    t = d[:, 0]
    t *=  1e-3 # convert timestamps from ms to s

    # observation data is g2(t) - 1
    g2_minus1_obs = d[:, 1:].T

    div = jax.vmap(lambda x: x/x[0])
    g1_squared = div(g2_minus1_obs) # normalization by first term handles removing the beta term (roughly)

    # g1 = jnp.sqrt(jnp.maximum(g1_squared, 0)) # this line converts to g1

    g1 = jnp.where(
        jnp.greater_equal(g1_squared, 0),
        jnp.sqrt(g1_squared),
        -jnp.sqrt(-g1_squared)
    )

    # g1 = jnp.sign(g1_squared) * jnp.sqrt(jnp.abs(g1_squared))

    theta = jnp.arange(30., 151, 5) # angles known in advance - in degrees
    q = scatter_vector(theta)
    return q, t, g1

def prep_data_g2(datafile):
    df = pd.read_csv(datafile, delimiter='\t', header=None)
    df = df.iloc[1:] # remove strange first point
    d = jnp.array(df.to_numpy())

    t = d[:, 0]
    t *=  1e-3 # convert timestamps from ms to s

    # observation data is g2(t) - 1
    g2_minus1_obs = d[:, 1:].T

    div = jax.vmap(lambda x: x/x[0])
    g1_squared = div(g2_minus1_obs) # normalization by first term handles removing the beta term (roughly)

    # # g1 = jnp.sqrt(jnp.maximum(g1_squared, 0)) # this line converts to g1
    # g1 = jnp.where(
    #     jnp.greater_equal(g1_squared, 0),
    #     jnp.sqrt(g1_squared),
    #     -jnp.sqrt(-g1_squared)
    # )

    theta = jnp.arange(30., 151, 5) # angles known in advance - in degrees
    q = scatter_vector(theta)
    # return q, t, g2_minus1_obs
    return q, t, g1_squared



In [7]:

######### Normal model ##########

def get_g1(t, nk, a, c):
    b = a**2/2
    d = jnp.sqrt(2)
    s_pi = jnp.sqrt(jnp.pi)
    x = (-a + c*t)/d
    y = jnp.where(
        jnp.greater_equal(x, 5),
        1/(x*s_pi),
        jax.scipy.special.erfc(x)*jnp.exp(x**2)
    )
    e = (nk/2)*jnp.exp(-b)
    return e*y

# vectorize along time dimension
all_g1 = jax.vmap(
    get_g1,
    in_axes=(0, None, None, None)
)

def source3_1(t, q, amp, mu, sig):
    ################
    const = SCALING_CONST
    # const = 1.0
    ################
    nk = amp
    sig = sig*const
    mu = mu*const
    a = mu/sig
    c = q**2*sig

    g = all_g1(t, nk, a, c)
    return g

by_Xs1 = jax.vmap(
    source3_1,
    in_axes=(None, None, 0, 0, 0)
)

source_matrix1 = jax.vmap(
    by_Xs1,
    in_axes=(None, 0, None, None, None)
)

def g1_matrix(q, t, amp, mu, sig):
    full = source_matrix1(t, q, amp, mu, sig)
    full = jnp.sum(full, axis=1)
    return full

def g2_minus1_matrix(q, t, amp, mu, sig, beta):
    g1 = g1_matrix(q, t, amp, mu, sig)
    g2_minus1 = beta * g1**2
    return g2_minus1


In [ ]:

# calculate normal curve values
def normal_distribution_single(x, amplitude, mu, sigma):
    return amplitude * jnp.exp(-(x-mu)**2/(2*sigma**2))/jnp.sqrt(2*jnp.pi*sigma**2)

normal_distributions = jax.vmap(normal_distribution_single, in_axes=(None, 0, 0, 0))

def normal_distribution(possible_D, amp, mu, sig):
    whole = normal_distributions(possible_D, amp, mu, sig).sum(axis=0)
    return whole / jnp.sum(whole) # normalize for plotting


In [ ]:
def g1_quadrature_(q, t, amp, mu, sig, scaling_const, laguerre_deg):
    x_lag, w_lag = orthax.laguerre.laggauss(laguerre_deg)
    # f_lag = normal_distributions(scaling_const * x_lag, amp, mu*SCALING_CONST, sig*SCALING_CONST)
    # return scaling_const * jnp.sum(w_lag * f_lag * jnp.exp((1-scaling_const*q**2*t) * x_lag))

    f_lag = normal_distributions(x_lag*scaling_const, amp, mu, sig)
    # f_lag = normal_distributions(scaling_const * x_lag, amp, mu*SCALING_CONST, sig*SCALING_CONST)
    # f_lag = normal_distributions(x_lag*scaling_const, amp, mu, sig)/SCALING_CONST_2
    return scaling_const * jnp.sum(w_lag * f_lag * jnp.exp((1-SCALING_CONST*scaling_const*q**2*t) * x_lag))


g1_quadrature_by_t = jax.vmap(g1_quadrature_, in_axes = (0, None, None, None, None, None, None))
g1_quadrature_by_q = jax.vmap(g1_quadrature_by_t, in_axes = (None, 0, None, None, None, None, None))

def g2_minus1_quadrature(q, t, amp, mu, sig, beta, scaling_const, laguerre_deg):
    # amp = amp/jnp.sum(amp)
    return beta*jnp.square(g1_quadrature_by_q(q, t, amp, mu, sig, scaling_const, laguerre_deg).T)


g2_minus1_quadrature_scaled = jax.jit(functools.partial(
    g2_minus1_quadrature,
    scaling_const=relative_scaling,
    laguerre_deg = 70
))



In [ ]:
q, t, _ = prep_data_g2("~/repos/DLS/Experimental_data_083122/stock_100nm.csv")

true_params = {
    'beta': 1.0,
    'amp': jnp.array([0.8, 0.2]),
    'mu': jnp.array([0.7, 0.1]),
    'sig': jnp.array([0.14, 0.03]),
    'loss': 0.01,
    'noise_std': 0.001
}

normal_obs = g2_minus1_matrix(
    q, t, true_params['amp'], true_params['mu'], true_params['sig'], true_params['beta']
)

quad_obs = g2_minus1_quadrature(q, t, true_params['amp'], true_params['mu'], true_params['sig'], true_params['beta'], relative_scaling, 70)


In [18]:
jnp.sum(jnp.abs(normal_obs - quad_obs))


Array(5.31875257e-05, dtype=float64)

In [22]:
diffusion_coef(5e-7)

4.905155346092215e-13